In [ ]:
# Parameters
lakehouse_name = "lh_medallion"
bronze_schema = "bronze"
mirrored_catalog_name = "mc_appareliq_uc"     # mirrored Azure Databricks catalog (Unity Catalog appareliq)
dbx_schema = "kinetiq_batch"                  # Unity Catalog schema inside the mirrored catalog
mirrored_database_name = "md_erp_wms"         # mirrored Azure SQL ERP/WMS database
retired_paths = "Files/landing,Files/code"    # leftovers of earlier designs, deleted if present

# 00 - lh_medallion: lakehouse, zero-copy Bronze shortcuts, cleanup

Everything that defines the `lh_medallion` lakehouse lives in this notebook; it is idempotent and runs first in
`pl_medallion`. No default lakehouse is attached (it may not exist yet): all work goes through the Fabric REST API and
OneLake with the identity running the notebook.

1. Create `lh_medallion` (schemas enabled) if it does not exist and wait for its SQL analytics endpoint.
2. Create or retarget the **zero-copy** Bronze shortcuts to the mirrored sources (no data is copied):
   * `bronze.dbx_<table>` -> `mc_appareliq_uc` (Databricks Unity Catalog `appareliq.kinetiq_batch`, DS-01, DS-02)
   * `bronze.<schema>_<table>` -> `md_erp_wms` (Azure SQL ERP/WMS, DS-03..10, DS-19)

   A physical table left at a shortcut path (an earlier copy) is deleted first.
3. Delete retired paths. Silver/Gold schemas and tables are created by notebooks 03/04; the Bronze file feeds by 01.

In [ ]:
import time
import pandas as pd
import requests

FABRIC = "https://api.fabric.microsoft.com/v1"
ONELAKE = "https://onelake.dfs.fabric.microsoft.com"
wsid = notebookutils.runtime.context["currentWorkspaceId"]


DBX_TABLES = ("sell_through_daily", "digital_intent_daily")
SQL_TABLES = ("ref.market", "ref.person", "ref.decision_owner", "ref.synthetic_run", "erp.location",
              "erp.product_style", "erp.product_sku", "erp.price", "erp.vendor", "erp.wholesale_partner",
              "erp.wholesale_order", "erp.wholesale_order_line", "erp.purchase_order", "erp.purchase_order_line",
              "erp.margin_guardrail", "erp.regional_event", "wms.inventory_position", "wms.transfer_lane",
              "wms.inventory_movement")
SQL_OPTIONAL_TABLES = ("erp.allocation_decision", "erp.stock_transfer_order")   # write-back tables, may be empty


def _h(audience):
    return {"Authorization": "Bearer " + notebookutils.credentials.getToken(audience)}


def api(method, path, body=None):
    """Fabric REST call; follows long-running operations (202 + Location) to their result."""
    for attempt in range(6):
        r = requests.request(method, FABRIC + path, headers=_h("pbi"), json=body, timeout=120)
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(int(r.headers.get("Retry-After", 10 * (attempt + 1))))
            continue
        break
    if r.status_code == 202 and "Location" in r.headers:
        while True:
            time.sleep(int(r.headers.get("Retry-After", 5)))
            r = requests.get(r.headers["Location"], headers=_h("pbi"), timeout=120)
            st = r.json() if r.content else {}
            if st.get("status") in ("Succeeded", "Failed", "Undefined"):
                if st["status"] != "Succeeded":
                    raise RuntimeError(f"{method} {path}: {st}")
                res = requests.get(r.headers["Location"] + "/result", headers=_h("pbi"), timeout=120)
                return res.json() if res.content else {}
            if r.status_code == 200 and "status" not in st:
                return st
    if r.status_code >= 400:
        raise RuntimeError(f"{method} {path} -> {r.status_code}: {r.text[:500]}")
    return r.json() if r.content else {}


def items():
    out, token = [], None
    while True:
        page = api("GET", f"/workspaces/{wsid}/items" + (f"?continuationToken={token}" if token else ""))
        out += page.get("value", [])
        token = page.get("continuationToken")
        if not token:
            return out


def item_id(name, types):
    found = [i for i in items() if i["displayName"] == name and i["type"] in types]
    assert found, f"{'/'.join(types)} '{name}' not found in workspace {wsid}"
    return found[0]["id"]

In [ ]:
# 1. Lakehouse (schemas enabled)
lh = next((i for i in items() if i["displayName"] == lakehouse_name and i["type"] == "Lakehouse"), None)
if lh is None:
    lh = api("POST", f"/workspaces/{wsid}/lakehouses", {
        "displayName": lakehouse_name,
        "description": "Medallion Bronze/Silver/Gold for UC1 inventory rebalancing - SYNTHETIC data",
        "creationPayload": {"enableSchemas": True}})
    print(f"created lakehouse {lakehouse_name}: {lh['id']}")
lh_id = lh["id"]
for _ in range(60):
    props = api("GET", f"/workspaces/{wsid}/lakehouses/{lh_id}").get("properties", {})
    status = props.get("sqlEndpointProperties", {}).get("provisioningStatus")
    if status == "Success":
        break
    print("SQL analytics endpoint:", status)
    time.sleep(10)
assert status == "Success", f"SQL analytics endpoint of {lakehouse_name} not provisioned: {status}"
print(f"{lakehouse_name}: {lh_id} (SQL endpoint ready)")

In [ ]:
# 2. Zero-copy Bronze shortcuts to the mirrored items
catalog_id = item_id(mirrored_catalog_name, ("MirroredAzureDatabricksCatalog",))
database_id = item_id(mirrored_database_name, ("MirroredDatabase",))
want = {f"dbx_{t}": (catalog_id, f"Tables/{dbx_schema}/{t}") for t in DBX_TABLES}
want |= {t.replace(".", "_"): (database_id, "Tables/" + t.replace(".", "/")) for t in SQL_TABLES + SQL_OPTIONAL_TABLES}

existing = {s["name"]: s for s in api("GET", f"/workspaces/{wsid}/items/{lh_id}/shortcuts").get("value", [])
            if s.get("path", "").strip("/") == f"Tables/{bronze_schema}"}
actions = []
for name, (target_item, target_path) in want.items():
    cur = existing.get(name, {}).get("target", {}).get("oneLake", {})
    if cur.get("itemId") == target_item and cur.get("path") == target_path:
        actions.append((name, "ok"))
        continue
    if name not in existing:   # a physical table at this path would hide the shortcut: drop the copy
        r = requests.delete(f"{ONELAKE}/{wsid}/{lh_id}/Tables/{bronze_schema}/{name}", params={"recursive": "true"},
                            headers=_h("storage"), timeout=120)
        if r.status_code not in (200, 404):
            raise RuntimeError(f"delete {bronze_schema}.{name} -> {r.status_code}: {r.text[:300]}")
        if r.status_code == 200:
            print(f"dropped copied table {bronze_schema}.{name}")
    api("POST", f"/workspaces/{wsid}/items/{lh_id}/shortcuts?shortcutConflictPolicy=CreateOrOverwrite",
        {"path": f"Tables/{bronze_schema}", "name": name,
         "target": {"oneLake": {"workspaceId": wsid, "itemId": target_item, "path": target_path}}})
    actions.append((name, "created" if name not in existing else "retargeted"))
print(pd.Series([a for _, a in actions]).value_counts().to_dict())

In [ ]:
# 3. Retired paths
for path in [p.strip() for p in retired_paths.split(",") if p.strip()]:
    r = requests.delete(f"{ONELAKE}/{wsid}/{lh_id}/{path}", params={"recursive": "true"},
                        headers=_h("storage"), timeout=120)
    if r.status_code not in (200, 404):
        raise RuntimeError(f"delete {path} -> {r.status_code}: {r.text[:300]}")
    print(f"{lakehouse_name}/{path}:", "deleted" if r.status_code == 200 else "not present")

In [ ]:
# Validation: every Bronze shortcut exists and points at the right mirrored table
now = {s["name"]: s["target"].get("oneLake", {}) for s in api("GET", f"/workspaces/{wsid}/items/{lh_id}/shortcuts")
       .get("value", []) if s.get("path", "").strip("/") == f"Tables/{bronze_schema}"}
check = pd.DataFrame([dict(shortcut=f"{bronze_schema}.{n}", target=f"{'dbx' if i == catalog_id else 'erp_wms'}:{p}",
                           ok=now.get(n, {}).get("itemId") == i and now.get(n, {}).get("path") == p)
                      for n, (i, p) in want.items()])
display(check)
assert check.ok.all(), check[~check.ok]
print(f"{lakehouse_name} ready: {len(check)} zero-copy Bronze shortcuts")